# Assignment #2: UNO Game AI
**Faculty:** Ms. Umarah Qaseem  
**Submitted by:** Muhammad Saad Sabir | 24i-2547

## GitHub Repository
**Link:** https://github.com/saad2547/UNO-AI-Assignment2

## 1. Imports & Setup

In [31]:
import random
import math

## 2. Card Class

In [32]:
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def is_skip(self):
        #Returns True if this is a Skip card.
        return self.value == 'Skip'

    def matches(self, top_card):
        """ Rule 1  Valid Move:
        A card is valid if it shares the same color OR same number as topCard.
        """
        return self.color == top_card.color or self.value == top_card.value

## 3. Deck Generator

In [33]:
def generate_deck():
    """
    Generates and shuffles a full UNO simplified deck.
    Contains:
      Red 0-9, Blue 0-9, Green 0-9, Yellow 0-9
      One Skip card per color so total 4 skip cards
      Returns a shuffled list of Card objects.
    """
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    # Number cards 0 to 9 for each color
    for color in colors:
        for number in range(10):
            deck.append(Card(color, number))

    # One Skip per color
    for color in colors:
        deck.append(Card(color, 'Skip'))

    random.shuffle(deck)
    return deck

## 4. Legal Move Generator

In [34]:
def get_valid_moves(hand, top_card):
    """
    Returns a list of cards from 'hand' that are valid plays
    given the current 'top_card' on the discard pile.
    Rule 1: same color OR same number.
    """
    return [card for card in hand if card.matches(top_card)]

## 5. Game State Representation

In [35]:
def create_initial_state():
    """
    Creates the initial game state.
    State dictionary:
      hands     : list of 3 hands 5 cards each
      top_card  : current top card
      deck      : remaining draw deck
      current   : index of the player whose turn it is (0, 1, or 2)
      skip_next : True if the next player's turn must be skipped
    """
    deck = generate_deck()

    # Deal 5 cards to each of the 3 players
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())

    # Flip the top card make sure it is NOT a Skip to start
    top_card = deck.pop()
    while top_card.is_skip():
        deck.insert(0, top_card)      # put it back at bottom
        top_card = deck.pop()

    return {
        'hands': hands,
        'top_card': top_card,
        'deck': deck,
        'current': 0,
        'skip_next': False
    }

## 6. State Transition

In [36]:
def apply_move(state, player_idx, card):
    """
    Returns a NEW state after 'player_idx' plays 'card'.
    card=None means the player draws one card.

    Rules applied:
      Rule 2  Draw: If card is None, draw 1 from deck.
      Rule 3  Skip: If played card is Skip, next player is skipped.
      Rule 4  Win : If hand becomes empty, winner is set.
    """
    # Deep copy so original state is not mutated
    new_state = {
        'hands': [list(h) for h in state['hands']],
        'top_card': state['top_card'],
        'deck': list(state['deck']),
        'current': state['current'],
        'skip_next': False,
        'winner': state.get('winner', None)
    }

    if card is None:
        # Draw 1 card 
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state['hands'][player_idx].append(drawn)
    else:
        # Play the card
        new_state['hands'][player_idx].remove(card)
        new_state['top_card'] = card

        # Rule 3 Skip
        if card.is_skip():
            new_state['skip_next'] = True

        # Rule 4 Win check
        if len(new_state['hands'][player_idx]) == 0:
            new_state['winner'] = player_idx

    # Advance turn
    next_player = (player_idx + 1) % 3
    if new_state['skip_next']:
        next_player = (player_idx + 2) % 3  
        new_state['skip_next'] = False
    new_state['current'] = next_player

    return new_state

## 7. Evaluation Function

### Formula
$$\text{Score} = 50 - 5 \cdot C_{AI} + 2 \cdot C_{opp} + 3 \cdot S$$

Where:
- $C_{AI}$: Cards in the AI's hand (fewer cards = better)
- $C_{opp}$: Average cards in opponents' hands
- $S$: Number of Skip cards in AI's hand

### Weight Tuning
| Strategy | $C_{AI}$ weight | $C_{opp}$ weight | $S$ weight | Rationale |
|----------|----------------|-----------------|------------|----------|
| **Defensive (P1)** | 4 | 1 | 5 | Values keeping Skips high to stall opponents |
| **Offensive (P2)** | 7 | 3 | 2 | Aggressively sheds cards, less care for Skips |

In [37]:
def evaluate(state, player_idx, strategy='defensive'):
    """
    Evaluation function for a given state from the perspective of player_idx.

    strategy = 'defensive' weights tuned for P1
    strategy = 'offensive' weights tuned for P2

    Base formula: Score = base - w1*C_AI + w2*C_opp + w3*S
    """
    hand = state['hands'][player_idx]
    opp_indices = [i for i in range(3) if i != player_idx]
    opp_cards_avg = sum(len(state['hands'][i]) for i in opp_indices) / 2

    c_ai = len(hand)
    c_opp = opp_cards_avg
    skips = sum(1 for c in hand if c.is_skip())

    if strategy == 'defensive':
        # Tuned weights: punish own cards less, reward skips more
        score = 50 - 4 * c_ai + 1 * c_opp + 5 * skips
    else:  # offensive
        # Tuned weights: heavily punish own cards, reward opponents having more
        score = 50 - 7 * c_ai + 3 * c_opp + 2 * skips

    return score

## 8. Minimax Search 
## Player 1  Defensive

In [38]:
minimax_tree_log = []

def minimax(state, depth, player_idx, is_maximizing, ai_player, log=None, level=0):
    """
    Minimax Search – used by Player 1 (Defensive strategy).

    Parameters:
      state         : current game state
      depth         : remaining search depth (starts at 3)
      player_idx    : index of the player whose turn it is right now
      is_maximizing : True if this is the AI's turn
      ai_player     : index of the AI (to compute evaluation from its perspective)
      log           : optional list to collect tree nodes
      level         : current tree level (for indented printing)

    Returns:
      (best_score, best_move)
      best_move is None for leaf/terminal nodes.
    """
    # Terminal check: someone won or depth exhausted
    if state.get('winner') is not None or depth == 0:
        score = evaluate(state, ai_player, strategy='defensive')
        if log is not None:
            log.append({'level': level, 'player': player_idx,
                        'type': 'LEAF', 'score': score, 'move': None})
        return score, None

    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    # If no valid move, only option is draw
    moves = valid_moves if valid_moves else [None]

    if is_maximizing:
        best_score = -math.inf
        best_move = None
        for move in moves:
            new_state = apply_move(state, player_idx, move)
            next_player = new_state['current']
            next_is_max = (next_player == ai_player)
            score, _ = minimax(new_state, depth - 1, next_player,
                               next_is_max, ai_player, log, level + 1)
            if log is not None:
                log.append({'level': level, 'player': player_idx,
                            'type': 'MAX', 'score': score, 'move': move})
            if score > best_score:
                best_score = score
                best_move = move
        return best_score, best_move
    else:
        # Minimising node – opponent plays to minimise AI score
        best_score = math.inf
        best_move = None
        for move in moves:
            new_state = apply_move(state, player_idx, move)
            next_player = new_state['current']
            next_is_max = (next_player == ai_player)
            score, _ = minimax(new_state, depth - 1, next_player,
                               next_is_max, ai_player, log, level + 1)
            if log is not None:
                log.append({'level': level, 'player': player_idx,
                            'type': 'MIN', 'score': score, 'move': move})
            if score < best_score:
                best_score = score
                best_move = move
        return best_score, best_move


def minimax_decision(state, player_idx, depth=3, verbose=True):
    """
    Entry point: returns the best move for 'player_idx' using Minimax (depth=3).
    Also prints all considered moves and their scores if verbose=True.
    """
    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)
    moves = valid_moves if valid_moves else [None]

    if verbose:
        print(f"\n[Minimax] Player {player_idx+1} considering moves (depth={depth}):")
        print(f"  Top card: {top_card}")
        print(f"  Hand: {hand}")

    best_score = -math.inf
    best_move = None

    for move in moves:
        new_state = apply_move(state, player_idx, move)
        next_player = new_state['current']
        next_is_max = (next_player == player_idx)
        score, _ = minimax(new_state, depth - 1, next_player,
                           next_is_max, player_idx)
        move_label = str(move) if move else "Draw"
        if verbose:
            print(f"    Move: {move_label:20s}  Score: {score:.1f}")
        if score > best_score:
            best_score = score
            best_move = move

    if verbose:
        print(f"  => Best move: {best_move}  (score={best_score:.1f})")
    return best_move, best_score

## 9. Expectimax Search 
## Player 2  Offensive

In [39]:
def expectimax(state, depth, player_idx, ai_player, log=None, level=0):
    """
    Expectimax Search used by Player 2 Offensive strategy.

    Node types:
      MAX node    : AI's turn  maximise expected score
      CHANCE node : Draw action expected value over all drawable cards
      OPPONENT    : opponent plays a random legal move

    Parameters:
      state      : current game state
      depth      : remaining depth starts at 3
      player_idx : player whose turn it is now
      ai_player  : index of the Expectimax AI Player 2 = index 1

    Returns:
      expected_score
    """
    # Terminal
    if state.get('winner') is not None or depth == 0:
        score = evaluate(state, ai_player, strategy='offensive')
        if log is not None:
            log.append({'level': level, 'player': player_idx,
                        'type': 'LEAF', 'score': score, 'move': None})
        return score

    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if player_idx == ai_player:
        # MAX node 
        if not valid_moves:
            # Must draw CHANCE node
            return _chance_node(state, depth, player_idx, ai_player, log, level)

        best = -math.inf
        for move in valid_moves:
            new_state = apply_move(state, player_idx, move)
            score = expectimax(new_state, depth - 1,
                               new_state['current'], ai_player, log, level + 1)
            if log is not None:
                log.append({'level': level, 'player': player_idx,
                            'type': 'MAX', 'score': score, 'move': move})
            if score > best:
                best = score
        return best
    else:
        #OPPONENT node
        if not valid_moves:
            # Opponent draws
            new_state = apply_move(state, player_idx, None)
            return expectimax(new_state, depth - 1,
                              new_state['current'], ai_player, log, level + 1)

        # Treat opponent as playing uniformly at random among legal moves
        total = 0.0
        prob = 1.0 / len(valid_moves)
        for move in valid_moves:
            new_state = apply_move(state, player_idx, move)
            score = expectimax(new_state, depth - 1,
                               new_state['current'], ai_player, log, level + 1)
            total += prob * score
        if log is not None:
            log.append({'level': level, 'player': player_idx,
                        'type': 'OPP', 'score': total, 'move': None})
        return total


def _chance_node(state, depth, player_idx, ai_player, log=None, level=0):
    """
    CHANCE node for the draw action.
    Computes the expected score over all cards remaining in the deck.
    P card = 1 /deck for each card uniform.
    """
    deck = state['deck']
    if not deck:
        # No cards left just evaluate current state
        return evaluate(state, ai_player, strategy='offensive')

    total = 0.0
    prob = 1.0 / len(deck)
    seen = set()

    for drawn_card in deck:
        key = (drawn_card.color, drawn_card.value)
        if key in seen:
            # Avoid duplicate identical cards only differ in identity
            pass
        seen.add(key)
        # Simulate drawing this card
        new_state = {
            'hands': [list(h) for h in state['hands']],
            'top_card': state['top_card'],
            'deck': [c for c in state['deck'] if c is not drawn_card],
            'current': state['current'],
            'skip_next': state.get('skip_next', False),
            'winner': state.get('winner', None)
        }
        new_state['hands'][player_idx].append(drawn_card)
        next_player = (player_idx + 1) % 3
        new_state['current'] = next_player

        score = expectimax(new_state, depth - 1,
                           next_player, ai_player, log, level + 1)
        total += prob * score

    if log is not None:
        log.append({'level': level, 'player': player_idx,
                    'type': 'CHANCE', 'score': total, 'move': 'Draw'})
    return total


def expectimax_decision(state, player_idx, depth=3, verbose=True):
    """
    Entry point: returns the best move for 'player_idx' using Expectimax.
    Prints all considered moves and expected scores if verbose=True.
    """
    hand = state['hands'][player_idx]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if verbose:
        print(f"\n[Expectimax] Player {player_idx+1} considering moves (depth={depth}):")
        print(f"  Top card: {top_card}")
        print(f"  Hand: {hand}")

    best_score = -math.inf
    best_move = None

    if not valid_moves:
        # Only option is draw chance node
        score = _chance_node(state, depth - 1, player_idx, player_idx)
        if verbose:
            print(f"    Move: {'Draw':20s}  Expected score: {score:.2f}")
        return None, score

    for move in valid_moves:
        new_state = apply_move(state, player_idx, move)
        score = expectimax(new_state, depth - 1,
                           new_state['current'], player_idx)
        move_label = str(move)
        if verbose:
            print(f"    Move: {move_label:20s}  Expected score: {score:.2f}")
        if score > best_score:
            best_score = score
            best_move = move

    # Also consider draw as an alternative
    draw_score = _chance_node(state, depth - 1, player_idx, player_idx)
    if verbose:
        print(f"    Move: {'Draw':20s}  Expected score: {draw_score:.2f}")
    if draw_score > best_score:
        best_score = draw_score
        best_move = None

    if verbose:
        print(f"  => Best move: {best_move}  (expected score={best_score:.2f})")
    return best_move, best_score

## 10. Game Tree Printer

In [50]:
def print_game_tree(log, title="Game Tree", max_nodes=40):

    print(f"  {title}")

    node_symbols = {'MAX': '>', 'MIN': '<', 'CHANCE': '%', 'OPP': '@', 'LEAF': '*'}

    for i, node in enumerate(log[:max_nodes]):
        indent = '  ' * node['level']
        sym = node_symbols.get(node['type'], '?')
        move_str = str(node['move']) if node['move'] else 'Draw'
        print(f"{indent}{sym} [{node['type']}] P{node['player']+1} | "
              f"Move: {move_str:15s} | Score: {node['score']:.2f}")

    if len(log) > max_nodes:
        print(f"  ... ({len(log) - max_nodes} more nodes not shown)")
        print("\n")

In [51]:
print("  TEST 1: Minimax tree log (real game state)")
random.seed(42)
state = create_initial_state()
mm_log = []
minimax(state, depth=3, player_idx=0, is_maximizing=True, ai_player=0, log=mm_log)
print(f"  Total nodes logged: {len(mm_log)}")
print_game_tree(mm_log, title="Minimax Tree — Player 1 (Defensive)", max_nodes=15)

print("  TEST 2: Expectimax tree log (real game state)")
random.seed(42)
state = create_initial_state()
em_log = []
expectimax(state, depth=3, player_idx=1, ai_player=1, log=em_log)
print(f"  Total nodes logged: {len(em_log)}")
print_game_tree(em_log, title="Expectimax Tree — Player 2 (Offensive)", max_nodes=15)

print("  TEST 3: max_nodes truncation works correctly")
print_game_tree(mm_log, title="Truncation test (max_nodes=5)", max_nodes=5)
expected_truncated = len(mm_log) > 5
print(f"  Truncation message shown: {expected_truncated} " if expected_truncated else "  No truncation needed")

print("  TEST 4: All node types present in logs")
mm_types  = set(n['type'] for n in mm_log)
em_types  = set(n['type'] for n in em_log)
print(f"  Minimax  node types : {mm_types}")
print(f"  Expectimax node types: {em_types}")
print(f"  Minimax  has MAX  : {'MAX'  in mm_types}")
print(f"  Minimax  has MIN  : {'MIN'  in mm_types}")
print(f"  Minimax  has LEAF : {'LEAF' in mm_types}")
print(f"  Expectimax has MAX   : {'MAX'    in em_types}")
print(f"  Expectimax has OPP   : {'OPP'    in em_types}")
print(f"  Expectimax has LEAF  : {'LEAF'   in em_types}")

print("  TEST 5: Manual log  all symbols render correctly")
manual_log = [
    {'level': 0, 'player': 0, 'type': 'MAX',    'score': 48.0,  'move': None},
    {'level': 1, 'player': 1, 'type': 'MIN',    'score': 43.0,  'move': None},
    {'level': 1, 'player': 1, 'type': 'OPP',    'score': 41.5,  'move': None},
    {'level': 2, 'player': 2, 'type': 'CHANCE', 'score': 39.2,  'move': 'Draw'},
    {'level': 2, 'player': 2, 'type': 'LEAF',   'score': 36.0,  'move': None},
]
print_game_tree(manual_log, title="Symbol test", max_nodes=40)

  TEST 1: Minimax tree log (real game state)
  Total nodes logged: 16
  Minimax Tree — Player 1 (Defensive)
      * [LEAF] P1 | Move: Draw            | Score: 48.00
    < [MIN] P3 | Move: Red 1           | Score: 48.00
      * [LEAF] P1 | Move: Draw            | Score: 48.00
    < [MIN] P3 | Move: Red 2           | Score: 48.00
  < [MIN] P2 | Move: Red 7           | Score: 48.00
      * [LEAF] P1 | Move: Draw            | Score: 48.00
    < [MIN] P3 | Move: Blue 4          | Score: 48.00
  < [MIN] P2 | Move: Blue 5          | Score: 48.00
      * [LEAF] P1 | Move: Draw            | Score: 49.00
    < [MIN] P3 | Move: Draw            | Score: 49.00
  < [MIN] P2 | Move: Green 7         | Score: 49.00
> [MAX] P1 | Move: Blue 7          | Score: 48.00
      * [LEAF] P2 | Move: Draw            | Score: 47.50
    > [MAX] P1 | Move: Blue 7          | Score: 47.50
  < [MIN] P3 | Move: Blue 4          | Score: 47.50
  ... (1 more nodes not shown)


  TEST 2: Expectimax tree log (real game state